In [1]:
!git clone https://github.com/Musoye/team_AfriTherm_Code_V1

Cloning into 'team_AfriTherm_Code_V1'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 53 (delta 22), reused 40 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 197.76 KiB | 1.23 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [2]:
!ls

sample_data  team_AfriTherm_Code_V1


In [3]:
%cd team_AfriTherm_Code_V1

/content/team_AfriTherm_Code_V1


In [4]:
!pip install groq openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.3 MB/s eta 0:00:00


In [5]:
!ls

bonus_ai_workflow.py	      results			tvd_convertion.py
Lithostratigraphic_Data.xlsx  surface_system_design.py	Well_Path_Data.xlsx
power_calculation.py	      target_lithologies.csv
README.md		      ThermoGIS_Data.xlsx


In [6]:
%run -i tvd_convertion.py
%run -i power_calculation.py
%run -i surface_system_design.py

In [7]:
print("=" * 65)
print("  TVD CONVERSION PIPELINE")
print("  Geothermal Challenge — Utrecht Rotliegend Reservoir")
print("=" * 65)

print("\n[Step 1] Loading well path survey tables...")
well_paths = load_well_paths(WELL_PATH_FILE)
for name, wp in well_paths.items():
    print(f"  {name}: {len(wp)} survey stations | "
          f"AH range {wp['Depth (m)'].max():.0f} m | "
          f"TVD range {wp['TVD (m)'].max():.0f} m | "
          f"max deviation {wp['Depth (m)'].max() - wp['TVD (m)'].max():.0f} m")

print("\n[Step 2] Converting Slochteren AH boundaries to TVD...")
tvd_bounds = get_slochteren_tvd_bounds(LITHO_FILE, well_paths)
for w, b in tvd_bounds.items():
    print(f"  {w}: AH {b['ah_top']:.1f}-{b['ah_base']:.1f} m → "
          f"TVD {b['tvd_top']:.1f}-{b['tvd_base']:.1f} m  "
          f"[correction: {b['correction_m']:.1f} m]")

print("\n[Step 3] Filling depth_tvd_m in target_lithologies.csv...")
df_out = fill_tvd_in_csv(CSV_FILE, tvd_bounds, OUTPUT_FILE)

print("[Step 4] Summary of corrected data:")
print_summary(df_out)

print(f"\nOutput saved → {OUTPUT_FILE}")
print("Next step: run power_calculation.py on this corrected file.")

  TVD CONVERSION PIPELINE
  Geothermal Challenge — Utrecht Rotliegend Reservoir

[Step 1] Loading well path survey tables...
  BLT-01: 102 survey stations | AH range 2123 m | TVD range 2051 m | max deviation 72 m
  EVD-01: 21 survey stations | AH range 2198 m | TVD range 2181 m | max deviation 16 m
  JUT-01: 33 survey stations | AH range 3409 m | TVD range 3325 m | max deviation 84 m
  PKP-01: 110 survey stations | AH range 2751 m | TVD range 2404 m | max deviation 347 m

[Step 2] Converting Slochteren AH boundaries to TVD...
  BLT-01: AH 1924.0-2052.7 m → TVD 1862.5-1984.9 m  [correction: 61.5 m]
  JUT-01: AH 1659.5-1787.0 m → TVD 1655.3-1781.0 m  [correction: 4.2 m]
  EVD-01: AH 1788.0-1866.0 m → TVD 1782.8-1859.6 m  [correction: 5.2 m]
  PKP-01: AH 2530.5-2603.5 m → TVD 2207.2-2271.6 m  [correction: 323.3 m]

[Step 3] Filling depth_tvd_m in target_lithologies.csv...

CSV loaded: 3455 rows, all depth_tvd_m currently NaN

  BLT-01:
    AH depth:     1924.0 – 2052.7 m  (thickness 128.7

In [8]:

  print("Loading ThermoGIS data...")
  wells_data = load_thermogis(THERMOGIS_FILE)
  print(f"Loaded {len(wells_data)} wells: {', '.join(wells_data.keys())}")

  print("\nCalculating power for all wells and scenarios...")
  results = assess_wells(wells_data)

  df_results, totals = combined_analysis(results)

  save_results(df_results, OUTPUT_CSV)

  print("\n" + "=" * 65)
  print("  CHALLENGE 1 CONCLUSION")
  print("=" * 65)
  print(f"""
The Rotliegend reservoir in the Utrecht area is PARTIALLY viable
for geothermal heating at neighbourhood scale.

Two wells (BLT-01 and JUT-01) can supply ~{totals['P50']['total_mw']:.1f} MW at P50,
which is {HEATING_TARGET_MW - totals['P50']['total_mw']:.1f} MW short of the {HEATING_TARGET_MW} MW target.

A heat pump (COP {HEAT_PUMP_COP}) requiring only {totals['P50']['gap_mw']/HEAT_PUMP_COP:.2f} MW of
electricity bridges the gap completely.

The 5 MW cooling demand is met via the injection loop and chiller.

This is a HYBRID GEOTHERMAL system — not a failure to meet the
target, but the standard real-world design for this type of project.
  """)

Loading ThermoGIS data...
Loaded 4 wells: BLT-01, EVD-01, JUT-01, PKP-01

Calculating power for all wells and scenarios...

  GEOTHERMAL POWER ASSESSMENT — UTRECHT ROTLIEGEND

Per-well power output (MW):

  Well        Temp °C  Perm P50 mD   P90 MW   P50 MW   P10 MW  Status
  ---------------------------------------------------------------
  BLT-01           77           82      0.9      5.7     25.6  Good contributor
  EVD-01           72            6      0.0      0.0      0.0  No flow — rock too tight
  JUT-01           72           40      1.3      2.7      5.4  Partial contributor
  PKP-01           88            1      0.0      0.0      0.0  No flow — rock too tight

Combined total across all wells:

  Scenario     Total MW     vs 10 MW target     Gap MW
  ----------------------------------------------------
  P90               2.2        below target  7.8 MW short
  P50               8.4        below target  1.6 MW short
  P10              31.0        MEETS target  21.0 MW surplu

In [9]:
 print_design_report()

import csv
rows = [
    ["Component",         "Parameter",             "Value",   "Unit"],
    ["Geothermal",        "Supply P50",             GEO_SUPPLY_MW, "MW"],
    ["Heat Pump",         "Output",                 HEAT_PUMP_OUTPUT_MW, "MW"],
    ["Heat Pump",         "Electricity input",      round(HEAT_PUMP_ELEC_INPUT_MW,2), "MW"],
    ["Heat Pump",         "COP",                    HEAT_PUMP_COP, "-"],
    ["Chiller",           "Free cooling",           COOLING_FREE_COOLING_MW, "MW"],
    ["Chiller",           "Absorption cooling",     COOLING_CHILLER_MW, "MW"],
    ["Chiller",           "Total cooling",          COOLING_TOTAL_MW, "MW"],
    ["Thermal Storage",   "Capacity",               STORAGE_ENERGY_MWH, "MWh"],
    ["Thermal Storage",   "Volume",                 STORAGE_VOLUME_M3, "m3"],
    ["Solar Thermal",     "Peak output",            SOLAR_THERMAL_PEAK_MW, "MW"],
    ["Solar Thermal",     "Area",                   SOLAR_THERMAL_AREA_M2, "m2"],
    ["Economics",         "Total CAPEX",            round(CAPEX_TOTAL_EUR/1e6,2), "M EUR"],
    ["Economics",         "Annual OPEX",            round(OPEX_TOTAL_EUR/1e3), "k EUR/yr"],
    ["Economics",         "Annual energy",          ANNUAL_TOTAL_MWH, "MWh/yr"],
    ["Economics",         "LCoE",                   round(LCOE_EUR_PER_MWH,1), "EUR/MWh"],
    ["Economics",         "Project lifetime",       PROJECT_LIFETIME_YEARS, "years"],
]
with open("surface_system_design.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)
print("Results saved → surface_system_design.csv")

  CHALLENGE 2 — SURFACE SYSTEM DESIGN REPORT
  Utrecht Rotliegend Geothermal — Neighbourhood Scale

SYSTEM OVERVIEW
───────────────────────────────────────────────────────────────
  Geothermal base supply:  8.4 MW  (BLT-01 + JUT-01, P50)
  Heating target:          10.0 MW
  Cooling target:           5.0 MW
  Heating gap to bridge:    1.6 MW  → filled by heat pump
  Cooling gap to bridge:    5.0 MW  → filled by chiller + free cooling

COMPONENT 1 — HEAT PUMP
───────────────────────────────────────────────────────────────
  Problem it solves:
    Geothermal gives 8.4 MW but target is 10.0 MW.
    The 1.6 MW gap must be bridged without drilling a new well.

  Why a heat pump:
    A heat pump moves heat rather than creating it.
    For 0.40 MW of electricity in, you get 1.6 MW of heat out.
    COP = 4.0 (industry standard for water-water systems at this temp range)

  Specification:
    Type:            Water-to-water heat pump
    Heat output:     1.6 MW
    Electricity:     0.40 MW input

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "" #replace me with the key you got from Groq

In [11]:
%run -i bonus_ai_workflow.py

In [12]:
main()


GEOTHERMAL DESIGN AUTOMATION PIPELINE
────────────────────────────────────────────────────────────
STEP 1/3 — Loading and processing well data
────────────────────────────────────────────────────────────
  BLT-01: AH 1924m → TVD 1862m (correction 62m)
  JUT-01: AH 1660m → TVD 1655m (correction 4m)
  EVD-01: AH 1788m → TVD 1783m (correction 5m)
  PKP-01: AH 2530m → TVD 2207m (correction 323m)

────────────────────────────────────────────────────────────
STEP 2/3 — Calculating thermal power per well
────────────────────────────────────────────────────────────
  BLT-01: 77.0°C | flow P50=105 m³/h | power P50=5.7 MW
  EVD-01: 72.0°C | flow P50=0 m³/h | power P50=0.0 MW
  JUT-01: 72.0°C | flow P50=55 m³/h | power P50=2.7 MW
  PKP-01: 88.0°C | flow P50=0 m³/h | power P50=0.0 MW

────────────────────────────────────────────────────────────
STEP 3/3 — Computing surface system design and LCoE
────────────────────────────────────────────────────────────
  Geo supply P50: 8.4 MW
  Heating gap: 1

In [13]:
!ls

ai_generated_report.txt       surface_system_design.csv
bonus_ai_workflow.py	      surface_system_design.py
geothermal_assessment.csv     target_lithologies.csv
Lithostratigraphic_Data.xlsx  target_lithologies_tvd_corrected.csv
pipeline_results.csv	      ThermoGIS_Data.xlsx
power_calculation.py	      tvd_convertion.py
README.md		      Well_Path_Data.xlsx
results


In [14]:
from google.colab import files
# Downloading our results

for file in [
    "ai_generated_report.txt",
    "target_lithologies_tvd_corrected.csv",
    "geothermal_assessment.csv",
    "surface_system_design.csv",
]:
    files.download(file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>